# StochasticEoS
FC-Tanh network (2 × 200 hidden units, tanh) trained with MSE loss on CIFAR-10.

In [3]:
import os
from google.colab import drive

drive.mount('/content/drive')
os.chdir('/content/drive/My Drive/Research/StochasticEoS')

Mounted at /content/drive


In [4]:
import torch
import torch.optim as optim
import sys
sys.path.insert(0, '.')

from model import FCTanh
from data import get_cifar10_loaders
from train import train_epoch, evaluate
from utils import get_device

## Hyperparameters

In [ ]:
LR         = 0.01
BATCH_SIZE = 128
EPOCHS     = 100
VERBOSE    = 5       # print progress every this many epochs
DATA_DIR   = '/content/drive/My Drive/Research/data/'

## Setup

In [8]:
device = get_device()
print(f'Using device: {device}')

train_loader, test_loader = get_cifar10_loaders(DATA_DIR, batch_size=BATCH_SIZE)

model = FCTanh().to(device)
optimizer = optim.SGD(model.parameters(), lr=LR)

Using device: cuda


100%|██████████| 170M/170M [00:13<00:00, 12.9MB/s]


## Training

In [ ]:
history = {'train_loss': [], 'train_acc': [], 'test_loss': [], 'test_acc': []}

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, device)
    te_loss, te_acc = evaluate(model, test_loader, device)

    history['train_loss'].append(tr_loss)
    history['train_acc'].append(tr_acc)
    history['test_loss'].append(te_loss)
    history['test_acc'].append(te_acc)

    if VERBOSE > 0 and epoch % VERBOSE == 0:
        print(f'Epoch {epoch:>{len(str(EPOCHS))}d}/{EPOCHS} | '
              f'Train loss {tr_loss:.4f}, acc {tr_acc:.3f} | '
              f'Test  loss {te_loss:.4f}, acc {te_acc:.3f}')

## Results

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history['train_loss'], label='train')
ax1.plot(history['test_loss'],  label='test')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('MSE Loss')
ax1.set_title('Loss'); ax1.legend()

ax2.plot(history['train_acc'], label='train')
ax2.plot(history['test_acc'],  label='test')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy'); ax2.legend()

plt.tight_layout()
plt.show()

## Constrained Training

In [ ]:
from train import constraint_training

CONSTRAINED_STEPS = 100
CONSTRAINED_BATCH  = 128

init_params = {k: v.clone() for k, v in model.state_dict().items()}
dataset = train_loader.dataset

c_model, c_stats, c_loss_history = constraint_training(
    model, criterion=torch.nn.MSELoss(), dataset=dataset,
    eta=LR, num_steps=CONSTRAINED_STEPS, batch_size=CONSTRAINED_BATCH,
    init_params=init_params,
)

In [ ]:
NUM_RUNS = 10

all_stats = []
all_loss_histories = []

for run in range(NUM_RUNS):
    _, run_stats, run_loss_history = constraint_training(
        model, criterion=torch.nn.MSELoss(), dataset=dataset,
        eta=LR, num_steps=CONSTRAINED_STEPS, batch_size=CONSTRAINED_BATCH,
        init_params=init_params,
    )
    all_stats.append(run_stats)
    all_loss_histories.append(run_loss_history)
    print(f'Run {run + 1}/{NUM_RUNS} done')